In [17]:
import os
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

# Move to project root
os.chdir("/home/ryu/code/DL_project")

print("CWD:", os.getcwd())
print("Files:", os.listdir("."))


CWD: /home/ryu/code/DL_project
Files: ['.gitignore', 'outputs', '.git', 'VENV', '.venv', 'requirements.txt', 'models', 'src', 'datasets', 'notebooks']


In [ ]:
# MODEL_PATH = "models/mobilenetv2_bt_mri.h5"
MODEL_PATH = "models/resnet50_v2.h5"
assert os.path.exists(MODEL_PATH), "Model file not found"

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,470 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2 (12.00 B)

In [19]:
dummy_input = tf.zeros((1, 224, 224, 3))
_ = model(dummy_input)


In [20]:
backbone = model.get_layer("mobilenetv2_1.00_224")
last_conv_layer = backbone.get_layer("Conv_1")

# Sanity check
for layer in backbone.layers[-10:]:
    print(layer.name)


block_16_expand_BN
block_16_expand_relu
block_16_depthwise
block_16_depthwise_BN
block_16_depthwise_relu
block_16_project
block_16_project_BN
Conv_1
Conv_1_bn
out_relu


In [21]:
# Model from input → last convolution layer
last_conv_model = tf.keras.models.Model(
    inputs=backbone.input,
    outputs=last_conv_layer.output
)

# Model from last conv output → final predictions
classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
x = classifier_input

# Use the classifier head of the original model
for layer in model.layers[1:]:
    x = layer(x)

classifier_model = tf.keras.models.Model(classifier_input, x)

# Force-build both models
_ = last_conv_model(dummy_input)
dummy_conv = tf.zeros((1,) + last_conv_layer.output.shape[1:])
_ = classifier_model(dummy_conv)


In [22]:
def preprocess_image(img_path):
    img = tf.keras.preprocessing.image.load_img(
        img_path, target_size=(224, 224)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    return img_array, img


In [23]:
def make_gradcam_heatmap(img_array, last_conv_model, classifier_model):
    conv_outputs = last_conv_model(img_array)

    with tf.GradientTape() as tape:
        tape.watch(conv_outputs)  #  REQUIRED
        predictions = classifier_model(conv_outputs)
        class_index = tf.argmax(predictions[0])
        loss = predictions[0, class_index]  # scalar

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap /= max_val

    return heatmap.numpy()


In [24]:
glioma_dir = "datasets/BT-MRI/test/glioma"
IMG_PATH = os.path.join(glioma_dir, os.listdir(glioma_dir)[0])

print("Using image:", IMG_PATH)
assert os.path.exists(IMG_PATH), "Image not found"


Using image: datasets/BT-MRI/test/glioma/BT-MRI Test GL (281).jpg


In [ ]:
# img_array, original_img = preprocess_image(IMG_PATH)

# heatmap = make_gradcam_heatmap(
#     img_array,
#     last_conv_model,
#     classifier_model
# )

In [26]:
# # Resize heatmap
# heatmap = cv2.resize(heatmap, (224, 224))
# heatmap = np.uint8(255 * heatmap)

# # Apply colormap
# heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

# # Convert original image for OpenCV
# original_img = np.array(original_img)
# original_bgr = cv2.cvtColor(original_img, cv2.COLOR_RGB2BGR)

# # Overlay
# superimposed_img = cv2.addWeighted(
#     original_bgr, 0.6, heatmap_color, 0.4, 0
# )

# # Save
# os.makedirs("outputs/gradcam", exist_ok=True)
# cv2.imwrite("outputs/gradcam/sample_gradcam.png", superimposed_img)

# # Display
# plt.figure(figsize=(6, 6))
# plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
# plt.axis("off")
# plt.show()


#### Multiple images


In [28]:
OUTPUT_DIR = "outputs/gradcam/mobilenet_grad"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [29]:

image_files = os.listdir(glioma_dir)[:10]   # 🔁 limit images

for img_name in image_files:
    img_path = os.path.join(glioma_dir, img_name)

    img_array, original_img = preprocess_image(img_path)

    heatmap = make_gradcam_heatmap(
        img_array,
        last_conv_model,
        classifier_model
    )

    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(
        heatmap, cv2.COLORMAP_JET
    )

    original_np = np.array(original_img)
    original_bgr = cv2.cvtColor(original_np, cv2.COLOR_RGB2BGR)

    superimposed = cv2.addWeighted(
        original_bgr, 0.6, heatmap_color, 0.4, 0
    )

    save_path = os.path.join(
        OUTPUT_DIR, f"mobilenet_grad_{img_name}"
    )
    cv2.imwrite(save_path, superimposed)

    print("Saved:", save_path)


Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (281).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (191).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (84).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (71).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (129).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (223).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (14).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (66).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (107).jpg
Saved: outputs/gradcam/mobilenet_grad/mobilenet_grad_BT-MRI Test GL (183).jpg
